# VERSION 7 — Fixed Cheek Extraction + Classification + AdamW
## End-to-end pipeline
```
Skin patch extraction (cheek → forehead → blob fallback)
→ Class folders (MST_1/ ... MST_10/)
→ Stratified random split 70/15/15 (safe — identity stripped)
→ EfficientNet-B0, blocks 6+7+8 unfrozen
→ CrossEntropyLoss, AdamW, CosineAnnealingLR
→ WeightedRandomSampler + rare class augmentation
→ TTA evaluation + confusion matrix + ONNX export
```

# ✅ CELL 1 — Imports

In [ ]:
# !pip install mediapipe opencv-python scikit-image scikit-learn seaborn

import os
import gc
import urllib.request
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mediapipe as mp
import seaborn as sns
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from torchvision import models, transforms, datasets
from PIL import Image
from skimage import color as skcolor
from skimage import measure
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tqdm.auto import tqdm

# ✅ CELL 2 — Config

In [ ]:
# ── PATHS ──
BASE_DIR     = "Source Dataset MST PATH"
CSV_PATH     = os.path.join(BASE_DIR, "MST LABEL.csv")
CLASS_DIR    = "Segmented MST Path"
SAVE_WEIGHTS = "best_model.pth"
ONNX_PATH    = "best_web.onnx"
CKPT_PREFIX  = "checkpoint"
PLOT_PATH    = "metrics.png"
CM_PATH      = "confusion_matrix.png"

# ── HYPERPARAMS ──
TARGET_SIZE    = 224
BATCH_SIZE     = 32
EPOCHS         = 200
WARMUP_EPOCHS  = 3
PATIENCE       = 15
NUM_CLASSES    = 10
RARE_THRESHOLD = 100
N_TTA          = 5
MIN_SKIN_PX    = 500

# ── LR ──
LR_BLOCK6 = 1e-5
LR_BLOCK7 = 2e-5
LR_BLOCK8 = 5e-5
LR_HEAD   = 1e-4
WEIGHT_DECAY = 1e-3

# ── SKIN LAB RANGE ──
SKIN_L_MIN, SKIN_L_MAX = 20, 90
SKIN_A_MIN, SKIN_A_MAX = 3,  25
SKIN_B_MIN, SKIN_B_MAX = 3,  40

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
os.makedirs(CLASS_DIR, exist_ok=True)

# ✅ CELL 3 — MediaPipe Setup + Extraction Logic

In [ ]:
model_url  = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
model_path = "face_landmarker.task"
if not os.path.exists(model_path):
    print("Downloading MediaPipe model...")
    urllib.request.urlretrieve(model_url, model_path)

mp_base    = python.BaseOptions(model_asset_path=model_path)
mp_options = vision.FaceLandmarkerOptions(base_options=mp_base, num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(mp_options)

LEFT_CHEEK_IDX  = [116, 123, 147, 213, 192, 214, 210, 211, 32, 208, 206, 203]
RIGHT_CHEEK_IDX = [345, 352, 376, 433, 416, 434, 430, 431, 262, 428, 426, 423]
FOREHEAD_IDX    = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
                   67, 109, 10, 151, 9, 8, 107, 66, 105, 63, 70]
NOSE_TIP=1; LEFT_EAR=234; RIGHT_EAR=454


def get_landmarks(path):
    try:
        result = landmarker.detect(mp.Image.create_from_file(path))
        if result.face_landmarks: return result.face_landmarks[0]
    except: pass
    return None

def face_orientation(lm):
    n=lm[NOSE_TIP]; l=lm[LEFT_EAR]; r=lm[RIGHT_EAR]
    dl=abs(n.x-l.x); dr=abs(n.x-r.x); total=dl+dr
    if total<1e-6: return 'frontal'
    if abs(dr-dl)/total < 0.15: return 'frontal'
    return 'right' if dl<dr else 'left'

def lm_pts(lm, idx, h, w):
    return np.array([[int(lm[i].x*w),int(lm[i].y*h)] for i in idx if i<len(lm)],dtype=np.int32)

def make_mask(pts, h, w):
    if len(pts)<3: return None
    hull=cv2.convexHull(pts); mask=np.zeros((h,w),dtype=np.uint8)
    cv2.fillConvexPoly(mask,hull,255); return mask

def skin_filter(img_np, region_mask):
    lab=skcolor.rgb2lab(img_np); L,A,B=lab[:,:,0],lab[:,:,1],lab[:,:,2]
    skin=((L>=SKIN_L_MIN)&(L<=SKIN_L_MAX)&(A>=SKIN_A_MIN)&(A<=SKIN_A_MAX)&(B>=SKIN_B_MIN)&(B<=SKIN_B_MAX))
    combined=(region_mask>0)&skin; out=img_np.copy(); out[~combined]=0.0
    return out,combined

def to_patch(masked, valid):
    if valid.sum()<MIN_SKIN_PX: return None
    rows=np.any(valid,axis=1); cols=np.any(valid,axis=0)
    r0,r1=np.where(rows)[0][[0,-1]]; c0,c1=np.where(cols)[0][[0,-1]]
    crop=masked[r0:r1+1,c0:c1+1]; ch,cw=crop.shape[:2]
    pil=Image.fromarray((crop*255).astype(np.uint8))
    if ch>=TARGET_SIZE and cw>=TARGET_SIZE:
        return pil.resize((TARGET_SIZE,TARGET_SIZE),Image.BILINEAR)
    canvas=Image.new("RGB",(TARGET_SIZE,TARGET_SIZE),(0,0,0))
    canvas.paste(pil,((TARGET_SIZE-min(cw,TARGET_SIZE))//2,(TARGET_SIZE-min(ch,TARGET_SIZE))//2))
    return canvas

def extract_patch(image_path):
    """Returns (PIL 224x224, method) or (None, None)."""
    try:
        img=Image.open(image_path).convert("RGB").resize((512,512),Image.BILINEAR)
        img_np=np.array(img,dtype=np.float32)/255.0
    except: return None,None
    h,w=img_np.shape[:2]
    lm=get_landmarks(image_path)
    if lm is not None:
        orient=face_orientation(lm)
        # Priority 1: Both cheeks
        if orient=='frontal':
            lmsk=make_mask(lm_pts(lm,LEFT_CHEEK_IDX,h,w),h,w)
            rmsk=make_mask(lm_pts(lm,RIGHT_CHEEK_IDX,h,w),h,w)
            if lmsk is not None and rmsk is not None:
                m,s=skin_filter(img_np,np.maximum(lmsk,rmsk))
                p=to_patch(m,s)
                if p: return p,'both_cheeks'
        # Priority 2: Single cheek
        order=[RIGHT_CHEEK_IDX,LEFT_CHEEK_IDX] if orient=='left' else [LEFT_CHEEK_IDX,RIGHT_CHEEK_IDX]
        for cidx in order:
            msk=make_mask(lm_pts(lm,cidx,h,w),h,w)
            if msk is not None:
                m,s=skin_filter(img_np,msk); p=to_patch(m,s)
                if p: return p,'single_cheek'
        # Priority 3: Forehead
        fmsk=make_mask(lm_pts(lm,FOREHEAD_IDX,h,w),h,w)
        if fmsk is not None:
            m,s=skin_filter(img_np,fmsk); p=to_patch(m,s)
            if p: return p,'forehead'
    # Priority 4: Largest LAB blob
    lab=skcolor.rgb2lab(img_np); L,A,B=lab[:,:,0],lab[:,:,1],lab[:,:,2]
    skin=((L>=SKIN_L_MIN)&(L<=SKIN_L_MAX)&(A>=SKIN_A_MIN)&(A<=SKIN_A_MAX)&(B>=SKIN_B_MIN)&(B<=SKIN_B_MAX))
    if skin.sum()<MIN_SKIN_PX: return None,None
    labeled=measure.label(skin); regions=measure.regionprops(labeled)
    if not regions: return None,None
    lg=max(regions,key=lambda r:r.area); bm=(labeled==lg.label)
    masked=img_np.copy(); masked[~bm]=0.0
    p=to_patch(masked,bm)
    return (p,'blob_fallback') if p else (None,None)

print("✅ Extraction logic ready.")

# 🆕 CELL 4 — Build Class Folders (Run Once)
```
Segmented MST Path/
  MST_1/  *.jpg
  MST_2/  *.jpg
  ...
  MST_10/ *.jpg
```

In [ ]:
for mst in range(1, 11):
    os.makedirs(os.path.join(CLASS_DIR, f"MST_{mst}"), exist_ok=True)

existing = sum(len(os.listdir(os.path.join(CLASS_DIR,f"MST_{i}"))) for i in range(1,11))

if existing > 100:
    print(f"Cache exists ({existing} images). Delete {CLASS_DIR} to recompute.")
else:
    df_raw = pd.read_csv(CSV_PATH)
    df_raw = df_raw.dropna(subset=["image_ID","MST","subject_name","mask"])
    df_raw = df_raw[df_raw["mask"].astype(str)=='0']
    df_raw = df_raw[df_raw["MST"].between(1,10)]
    df_raw["MST"] = df_raw["MST"].astype(int)

    image_map = {}
    for root,_,files in os.walk(BASE_DIR):
        for f in files:
            if f.lower().endswith((".jpg",".jpeg",".png")):
                image_map[f] = os.path.join(root,f)

    df_raw["orig_path"] = df_raw["image_ID"].map(image_map)
    df_raw = df_raw.dropna(subset=["orig_path"]).reset_index(drop=True)
    print(f"Processing {len(df_raw)} images...")

    skipped=0; methods={"both_cheeks":0,"single_cheek":0,"forehead":0,"blob_fallback":0}

    for _,row in tqdm(df_raw.iterrows(),total=len(df_raw),desc="Building folders"):
        img_id    = os.path.splitext(row["image_ID"])[0]
        save_path = os.path.join(CLASS_DIR,f"MST_{row['MST']}",f"{img_id}.jpg")
        if os.path.exists(save_path): continue
        patch,method = extract_patch(row["orig_path"])
        if patch is None: skipped+=1; continue
        patch.save(save_path,"JPEG",quality=95)
        methods[method] = methods.get(method,0)+1

    print(f"\nSaved: {sum(methods.values())} | Skipped: {skipped}")
    print("Extraction breakdown:")
    for m,c in methods.items(): print(f"  {m:<15}: {c}")

print(f"\nClass distribution:")
for mst in range(1,11):
    n=len(os.listdir(os.path.join(CLASS_DIR,f"MST_{mst}")))
    print(f"  MST_{mst:>2}: {n}")

# ✅ CELL 5 — Dataset + Stratified Split + Augmentation

In [ ]:
full_dataset = datasets.ImageFolder(root=CLASS_DIR)
all_labels   = full_dataset.targets
all_indices  = list(range(len(full_dataset)))

print(f"Total  : {len(full_dataset)}")
print(f"Classes: {full_dataset.classes}")

# 70/15/15 stratified split
train_idx, temp_idx = train_test_split(all_indices, test_size=0.30, stratify=all_labels, random_state=42)
temp_labels = [all_labels[i] for i in temp_idx]
val_idx, test_idx   = train_test_split(temp_idx,    test_size=0.50, stratify=temp_labels, random_state=42)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

train_label_counts = Counter([all_labels[i] for i in train_idx])
rare_cls = {c for c,n in train_label_counts.items() if n < RARE_THRESHOLD}
print(f"Rare classes: {[full_dataset.classes[i] for i in sorted(rare_cls)]}")

standard_aug = transforms.Compose([
    transforms.Resize((TARGET_SIZE,TARGET_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15,contrast=0.15,saturation=0.0,hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
])
rare_aug_tf = transforms.Compose([
    transforms.Resize((TARGET_SIZE,TARGET_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.30,contrast=0.30,saturation=0.0,hue=0.0),
    transforms.RandomAffine(degrees=0,translate=(0.05,0.05),scale=(0.95,1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((TARGET_SIZE,TARGET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
])

class RareAwareSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, rare_cls, augment=False):
        self.dataset=dataset; self.indices=indices
        self.rare_cls=rare_cls; self.augment=augment
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        path,label=self.dataset.samples[self.indices[idx]]
        img=Image.open(path).convert("RGB")
        if self.augment:
            tf=rare_aug_tf if label in self.rare_cls else standard_aug
        else:
            tf=val_tf
        return tf(img),label

train_dataset = RareAwareSubset(full_dataset, train_idx, rare_cls, augment=True)
val_dataset   = RareAwareSubset(full_dataset, val_idx,   rare_cls, augment=False)
test_dataset  = RareAwareSubset(full_dataset, test_idx,  rare_cls, augment=False)

# WeightedRandomSampler
train_labels   = [all_labels[i] for i in train_idx]
cls_weights    = {c: 1.0/n for c,n in train_label_counts.items()}
sample_weights = [cls_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

# ✅ CELL 6 — Model + AdamW Optimizer

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze all
for param in model.parameters():
    param.requires_grad = False

# Unfreeze blocks 6+7+8
for block_idx in [6,7,8]:
    for param in model.features[block_idx].parameters():
        param.requires_grad = True

# Replace head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=True),
    nn.Linear(in_features, NUM_CLASSES)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

# AdamW — decoupled weight decay, better generalization than Adam
optimizer = torch.optim.AdamW([
    {"params": model.features[6].parameters(), "lr": LR_BLOCK6},
    {"params": model.features[7].parameters(), "lr": LR_BLOCK7},
    {"params": model.features[8].parameters(), "lr": LR_BLOCK8},
    {"params": model.classifier.parameters(),  "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS-WARMUP_EPOCHS, eta_min=1e-7
)

def calculate_accuracy(outputs, labels):
    _, preds = torch.max(outputs, 1)
    return (preds == labels).float().mean()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Optimizer       : AdamW  |  Weight decay: {WEIGHT_DECAY}")

# ✅ CELL 7 — Training Loop

In [ ]:
torch.cuda.empty_cache(); gc.collect()

best_val_acc=0.0; scaler=GradScaler(); early_stop_counter=0
history={"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[],"lr":[]}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # Warmup
    if epoch < WARMUP_EPOCHS:
        wf = (epoch+1)/WARMUP_EPOCHS
        for i,pg in enumerate(optimizer.param_groups):
            pg['lr'] = [LR_BLOCK6,LR_BLOCK7,LR_BLOCK8,LR_HEAD][i] * wf

    # --- TRAINING ---
    model.eval()
    for b in [6,7,8]: model.features[b].train()
    model.classifier.train()

    train_loss,train_acc = 0,0
    for images,labels in tqdm(train_loader,desc="Training"):
        try:
            images,labels = images.to(device),labels.to(device)
            optimizer.zero_grad()
            with autocast(device_type='cuda'):
                out  = model(images)
                loss = criterion(out,labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            train_loss += loss.item()
            train_acc  += calculate_accuracy(out,labels).item()
        except RuntimeError as e:
            if "allocation" in str(e) or "out of memory" in str(e):
                torch.cuda.empty_cache(); print("⚠️ OOM — skipped"); continue
            raise

    avg_train_loss = train_loss/len(train_loader)
    avg_train_acc  = train_acc/len(train_loader)

    if epoch >= WARMUP_EPOCHS: scheduler.step()
    current_lr = optimizer.param_groups[-1]['lr']

    # --- VALIDATION ---
    model.eval(); val_loss,val_acc = 0,0
    with torch.no_grad():
        for images,labels in tqdm(val_loader,desc="Validation"):
            images,labels = images.to(device),labels.to(device)
            with autocast(device_type='cuda'):
                out=model(images); loss=criterion(out,labels)
            val_loss += loss.item(); val_acc += calculate_accuracy(out,labels).item()

    avg_val_loss=val_loss/len(val_loader); avg_val_acc=val_acc/len(val_loader)

    history["train_loss"].append(avg_train_loss); history["val_loss"].append(avg_val_loss)
    history["train_acc"].append(avg_train_acc);   history["val_acc"].append(avg_val_acc)
    history["lr"].append(current_lr)

    print(f"LR: {current_lr:.2e}")
    print(f"Train - Loss: {avg_train_loss:.4f} | Acc: {avg_train_acc:.4f}")
    print(f"Val   - Loss: {avg_val_loss:.4f}  | Acc: {avg_val_acc:.4f}")

    if (epoch+1) % 5 == 0:
        ckpt=f"{CKPT_PREFIX}_epoch_{epoch+1}.pth"
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(),
                    'best_val_acc':best_val_acc,'early_stop_counter':early_stop_counter,
                    'history':history}, ckpt)
        print(f"💾 Checkpoint: {ckpt}")

    if avg_val_acc > best_val_acc:
        best_val_acc=avg_val_acc; early_stop_counter=0
        torch.save(model.state_dict(), SAVE_WEIGHTS)
        print(f"🔥 Saved best model! Val Acc: {best_val_acc:.4f}")
    else:
        early_stop_counter+=1
        print(f"No improvement. Early stop: {early_stop_counter}/{PATIENCE}")
        if early_stop_counter >= PATIENCE:
            print("⛔ Early stopping triggered."); break

print(f"\n✅ Done. Best val acc: {best_val_acc:.4f}")

# ✅ CELL 8 — Plot Training Metrics

In [ ]:
epochs_ran=len(history["train_loss"]); x=range(1,epochs_ran+1)
best_ep=history["val_acc"].index(max(history["val_acc"]))+1

fig=plt.figure(figsize=(18,5))
fig.suptitle("Version 7 — Classification Training Metrics",fontsize=14,fontweight='bold',y=1.02)
gs=gridspec.GridSpec(1,3,figure=fig,wspace=0.35)

ax1=fig.add_subplot(gs[0])
ax1.plot(x,history["train_loss"],label="Train",color="#2196F3",linewidth=2)
ax1.plot(x,history["val_loss"],  label="Val",  color="#F44336",linewidth=2,linestyle="--")
ax1.axvline(x=best_ep,color="gray",linestyle=":",linewidth=1.2,label=f"Best ({best_ep})")
ax1.axvspan(0,WARMUP_EPOCHS,alpha=0.08,color='green',label="Warmup")
ax1.set_title("Loss",fontweight='bold'); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(fontsize=8); ax1.grid(True,alpha=0.3)

ax2=fig.add_subplot(gs[1])
ax2.plot(x,history["train_acc"],label="Train",color="#4CAF50",linewidth=2)
ax2.plot(x,history["val_acc"],  label="Val",  color="#FF9800",linewidth=2,linestyle="--")
ax2.axvline(x=best_ep,color="gray",linestyle=":",linewidth=1.2)
ax2.axvspan(0,WARMUP_EPOCHS,alpha=0.08,color='green')
best_acc=max(history["val_acc"])
ax2.annotate(f"Best: {best_acc:.4f}",xy=(best_ep,best_acc),
             xytext=(best_ep+max(1,epochs_ran*0.05),best_acc-0.05),
             fontsize=9,color="#FF9800",
             arrowprops=dict(arrowstyle="->",color="#FF9800",lw=1.2))
ax2.set_title("Accuracy",fontweight='bold'); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_ylim(0,1); ax2.legend(fontsize=8); ax2.grid(True,alpha=0.3)

ax3=fig.add_subplot(gs[2])
ax3.plot(x,history["lr"],color="#607D8B",linewidth=2)
ax3.axvspan(0,WARMUP_EPOCHS,alpha=0.08,color='green',label="Warmup")
ax3.set_title("LR (head)",fontweight='bold'); ax3.set_xlabel("Epoch"); ax3.set_ylabel("LR")
ax3.legend(fontsize=8); ax3.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_PATH,dpi=150,bbox_inches='tight'); plt.show()
print(f"📊 Saved: {PLOT_PATH}")
print(f"Best val acc: {best_acc:.4f} at epoch {best_ep}")

# ✅ CELL 9 — Test Evaluation + TTA + Confusion Matrix

In [ ]:
model.load_state_dict(torch.load(SAVE_WEIGHTS,map_location=device,weights_only=True))
model.eval()
class_names=[f"MST_{i}" for i in range(1,11)]

# Standard test eval
std_preds,gt=[],[]
with torch.no_grad():
    for images,labels in tqdm(test_loader,desc="Standard eval"):
        images=images.to(device)
        with autocast(device_type='cuda'): out=model(images)
        _,preds=torch.max(out,1)
        std_preds.extend(preds.cpu().numpy()); gt.extend(labels.numpy())

std_acc=accuracy_score(gt,std_preds)
print(f"Standard Test Acc: {std_acc:.4f}")
print(classification_report(gt,std_preds,target_names=class_names))

# TTA eval
tta_tf=transforms.Compose([
    transforms.Resize((TARGET_SIZE,TARGET_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1,contrast=0.1,saturation=0.0,hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
])

tta_preds=[]
with torch.no_grad():
    for idx in tqdm(range(len(test_dataset)),desc=f"TTA (n={N_TTA})"):
        path,_=full_dataset.samples[test_idx[idx]]
        img=Image.open(path).convert("RGB")
        logits=None
        for i in range(N_TTA):
            t=val_tf(img) if i==0 else tta_tf(img)
            with autocast(device_type='cuda'): out=model(t.unsqueeze(0).to(device))
            logits=out if logits is None else logits+out
        _,pred=torch.max(logits,1); tta_preds.append(pred.item())

tta_acc=accuracy_score(gt,tta_preds)
print(f"TTA Test Acc (n={N_TTA}): {tta_acc:.4f}")
print(f"TTA improvement: {tta_acc-std_acc:.4f}")

# Confusion matrix
cm=confusion_matrix(gt,tta_preds)
fig,ax=plt.subplots(figsize=(10,8))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
            xticklabels=class_names,yticklabels=class_names,ax=ax)
ax.set_title(f"V7 Confusion Matrix (TTA) — Test Acc: {tta_acc:.4f}",fontweight='bold')
ax.set_ylabel("True"); ax.set_xlabel("Predicted")
plt.tight_layout(); plt.savefig(CM_PATH,dpi=150,bbox_inches='tight'); plt.show()
print(f"📊 Confusion matrix: {CM_PATH}")

# ✅ CELL 10 — ONNX Export

In [ ]:
model.load_state_dict(torch.load(SAVE_WEIGHTS,map_location=device,weights_only=True))
model.eval()

dummy=torch.randn(1,3,TARGET_SIZE,TARGET_SIZE,device=device)
torch.onnx.export(
    model,dummy,ONNX_PATH,
    export_params=True,opset_version=11,do_constant_folding=True,
    input_names=['input'],output_names=['output'],
    dynamic_axes={'input':{0:'batch_size'},'output':{0:'batch_size'}}
)
print(f"🚀 ONNX exported: {ONNX_PATH}")
print(f"\n⚠️  Inference pipeline:")
print(f"   1. Extract skin patch (cheek → forehead → blob)")
print(f"   2. Resize/pad to {TARGET_SIZE}×{TARGET_SIZE}")
print(f"   3. ToTensor() + Normalize(mean={IMAGENET_MEAN}, std={IMAGENET_STD})")
print(f"   4. ONNX outputs 10 logits → argmax + 1 = predicted MST")
print(f"   5. TTA n={N_TTA}: average softmax probs before argmax")